# LightGBM — .NET Files
Malware detection using static analysis features from EMBER2024.  
File type: `.NET` executables  
Model: LightGBM

## 1. Imports

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from lightgbm import LGBMClassifier, early_stopping

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, log_loss, roc_auc_score, roc_curve,
    average_precision_score
)

## 2. Config

In [ ]:
# ---- CHANGE THIS ----
BASE_PATH = r"C:\Users\makis\ember_data\dotnet_clean"
# ---------------------

DIM          = 2568
RANDOM_STATE = 13

# EMBER2024 feature group offsets
GENERAL_START,  GENERAL_END  = 0,    7      # GeneralFileInfo       (7)
HIST_START,     HIST_END     = 7,    263    # ByteHistogram         (256)
ENTROPY_START,  ENTROPY_END  = 263,  519    # ByteEntropyHistogram  (256)
STRING_START,   STRING_END   = 519,  696    # StringExtractor       (177)
HEADER_START,   HEADER_END   = 696,  770    # HeaderFileInfo        (74)
SECTION_START,  SECTION_END  = 770,  994    # SectionInfo           (224)
IMPORT_START,   IMPORT_END   = 994,  2276   # Imports               (1282)
EXPORT_START,   EXPORT_END   = 2276, 2405   # Exports               (129)
DATADIR_START,  DATADIR_END  = 2405, 2439   # DataDirectories       (34)
RICH_START,     RICH_END     = 2439, 2472   # RichHeader            (33)
AUTH_START,     AUTH_END     = 2472, 2480   # AuthenticodeSignature (8)
PEWARN_START,   PEWARN_END   = 2480, 2568   # PEFormatWarnings      (88)
DOS_START,      DOS_END      = 753,  770    # DOS Header (inside HeaderFileInfo)

## 3. Load Data

In [ ]:
X_train_full = np.memmap(os.path.join(BASE_PATH, "X_train.dat"), dtype=np.float32, mode="r").reshape(-1, DIM)
y_train      = np.memmap(os.path.join(BASE_PATH, "y_train.dat"), dtype=np.int32,   mode="r")

X_test_full  = np.memmap(os.path.join(BASE_PATH, "X_test.dat"),  dtype=np.float32, mode="r").reshape(-1, DIM)
y_test       = np.memmap(os.path.join(BASE_PATH, "y_test.dat"),  dtype=np.int32,   mode="r")

print(f"Train: {X_train_full.shape[0]:,} samples | Test: {X_test_full.shape[0]:,} | Features: {DIM}")
print("Train labels:", dict(pd.Series(y_train).value_counts()))
print("Test  labels:", dict(pd.Series(y_test).value_counts()))

## 4. Helpers

In [ ]:
CAT_FEATURES = [2, 3, 4, 5, 6, 701, 702]

def evaluate(model, X, y, label=""):
    t0 = time.perf_counter()
    proba = model.predict_proba(X)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    inf_time = time.perf_counter() - t0

    acc  = accuracy_score(y, pred)
    prec = precision_score(y, pred, zero_division=0)
    rec  = recall_score(y, pred, zero_division=0)
    f1   = f1_score(y, pred, zero_division=0)
    ll   = log_loss(y, proba)
    roc  = roc_auc_score(y, proba)
    pr   = average_precision_score(y, proba)
    cm   = confusion_matrix(y, pred)

    print(f"\n=== {label} ===")
    print(f"Inference time : {inf_time:.4f} s")
    print(f"Accuracy       : {acc:.6f}")
    print(f"Precision      : {prec:.6f}")
    print(f"Recall         : {rec:.6f}")
    print(f"F1             : {f1:.6f}")
    print(f"Log Loss       : {ll:.6f}")
    print(f"ROC-AUC        : {roc:.6f}")
    print(f"PR-AUC         : {pr:.6f}")
    print(f"Confusion Matrix:\n{cm}")

    fpr, tpr, _ = roc_curve(y, proba)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"{label} (AUC={roc:.4f})")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {label}")
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()
    return {"label": label, "accuracy": acc, "f1": f1, "roc_auc": roc, "pr_auc": pr}

def run_experiment(X_tr, X_te, y_tr, y_te, model, label, use_cat=False):
    print(f"\nFeatures: {X_tr.shape[1]} cols")
    t0 = time.perf_counter()
    if use_cat:
        model.fit(X_tr, y_tr, categorical_feature=CAT_FEATURES)
    else:
        model.fit(X_tr, y_tr)
    print(f"Training time: {time.perf_counter() - t0:.2f} s")
    return evaluate(model, X_te, y_te, label)

## 5. Baseline Model

In [ ]:
lgbm_baseline = LGBMClassifier(
    objective="binary",
    n_estimators=500,
    num_leaves=64,
    min_child_samples=100,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=-1
)

print("Training baseline LightGBM...")
t0 = time.perf_counter()
lgbm_baseline.fit(X_train_full, y_train, categorical_feature=CAT_FEATURES)
print(f"Training time: {time.perf_counter() - t0:.2f} s")

evaluate(lgbm_baseline, X_test_full, y_test, "LightGBM Baseline — Full Features")

## 6. Hyperparameter Tuning
### Phase 1 — Grid search on core params (early stopping)

In [ ]:
SUBSET_SIZE = 100_000
TRAIN_SIZE  = 95_000
VAL_SIZE    = 5_000

X_sub = X_train_full[:SUBSET_SIZE]
y_sub = y_train[:SUBSET_SIZE]

rng = np.random.RandomState(RANDOM_STATE)
X_tr = X_sub[:TRAIN_SIZE][rng.permutation(TRAIN_SIZE)]
y_tr = y_sub[:TRAIN_SIZE][rng.permutation(TRAIN_SIZE)]
X_val = X_sub[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]
y_val = y_sub[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE]

print("Train:", X_tr.shape, "| Val:", X_val.shape)

In [ ]:
N_ESTIMATORS        = 3000
EARLY_STOP_ROUNDS   = 50

results = []
run_id  = 0

for lr in [0.03, 0.05, 0.075]:
    for num_leaves in [63, 127, 255]:
        for min_child in [100, 200, 300]:
            run_id += 1
            params = {"learning_rate": lr, "num_leaves": num_leaves, "min_child_samples": min_child}
            print(f"\nRun #{run_id} | {params}")

            model = LGBMClassifier(
                objective="binary", n_estimators=N_ESTIMATORS,
                n_jobs=-1, random_state=RANDOM_STATE, **params
            )
            t0 = time.time()
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                      eval_metric="auc",
                      callbacks=[early_stopping(EARLY_STOP_ROUNDS, verbose=False)])
            train_time = time.time() - t0

            proba = model.predict_proba(X_val)[:, 1]
            results.append({**params,
                "best_n_estimators": model.best_iteration_,
                "roc_auc": roc_auc_score(y_val, proba),
                "pr_auc":  average_precision_score(y_val, proba),
                "train_time_s": train_time})

df_phase1 = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
print("\nTop 5 (Phase 1):")
print(df_phase1.head().to_string(index=False))

### Phase 2 — Regularization params

In [ ]:
FIXED_LR        = 0.05
FIXED_LEAVES    = 127
FIXED_MIN_CHILD = 200

results2 = []
run_id   = 0

for ff in [0.70, 0.85, 1.00]:
    for bf in [0.70, 0.85, 1.00]:
        for msg in [0.00, 0.05, 0.10]:
            run_id += 1
            tune_params = {"feature_fraction": ff, "bagging_fraction": bf,
                           "bagging_freq": 1, "min_split_gain": msg}
            print(f"\nRun #{run_id} | {tune_params}")

            model = LGBMClassifier(
                objective="binary", n_estimators=N_ESTIMATORS, n_jobs=-1,
                random_state=RANDOM_STATE,
                learning_rate=FIXED_LR, num_leaves=FIXED_LEAVES,
                min_child_samples=FIXED_MIN_CHILD, **tune_params
            )
            t0 = time.time()
            model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                      eval_metric="auc",
                      callbacks=[early_stopping(EARLY_STOP_ROUNDS, verbose=False)])
            train_time = time.time() - t0

            proba = model.predict_proba(X_val)[:, 1]
            results2.append({**tune_params,
                "best_n_estimators": model.best_iteration_,
                "roc_auc": roc_auc_score(y_val, proba),
                "pr_auc":  average_precision_score(y_val, proba),
                "train_time_s": train_time})

df_phase2 = pd.DataFrame(results2).sort_values("pr_auc", ascending=False)
print("\nTop 5 (Phase 2):")
print(df_phase2.head().to_string(index=False))

## 7. Tuned Model — Full Training & Evaluation

In [ ]:
BEST_PARAMS = {
    "n_estimators":     600,
    "num_leaves":       127,
    "min_child_samples":200,
    "learning_rate":    0.05,
    "feature_fraction": 0.70,
    "bagging_fraction": 0.85,
    "bagging_freq":     1,
    "min_split_gain":   0.05,
}

lgbm_tuned = LGBMClassifier(
    objective="binary", n_jobs=-1, random_state=RANDOM_STATE, verbose=-1,
    **BEST_PARAMS
)

print("Training tuned LightGBM on full training set...")
t0 = time.perf_counter()
lgbm_tuned.fit(X_train_full, y_train, categorical_feature=CAT_FEATURES)
print(f"Training time: {time.perf_counter() - t0:.2f} s")

evaluate(lgbm_tuned, X_test_full, y_test, "LightGBM Tuned — Full Features")

## 8. Feature Group Experiments

In [ ]:
def get_lgbm():
    return LGBMClassifier(objective="binary", n_jobs=-1,
                          random_state=RANDOM_STATE, verbose=-1, **BEST_PARAMS)

In [ ]:
# ByteHistogram + ByteEntropyHistogram
run_experiment(X_train_full[:, HIST_START:ENTROPY_END],
               X_test_full[:,  HIST_START:ENTROPY_END],
               y_train, y_test, get_lgbm(),
               "LightGBM — ByteHistogram + ByteEntropyHistogram")

In [ ]:
# Imports only
run_experiment(X_train_full[:, IMPORT_START:IMPORT_END],
               X_test_full[:,  IMPORT_START:IMPORT_END],
               y_train, y_test, get_lgbm(), "LightGBM — Imports only")

In [ ]:
# Exports only
run_experiment(X_train_full[:, EXPORT_START:EXPORT_END],
               X_test_full[:,  EXPORT_START:EXPORT_END],
               y_train, y_test, get_lgbm(), "LightGBM — Exports only")

In [ ]:
# New EMBER2024 features
X_tr_new = np.hstack([X_train_full[:, DOS_START:DOS_END], X_train_full[:, DATADIR_START:PEWARN_END]])
X_te_new = np.hstack([X_test_full[:,  DOS_START:DOS_END], X_test_full[:,  DATADIR_START:PEWARN_END]])
run_experiment(X_tr_new, X_te_new, y_train, y_test, get_lgbm(), "LightGBM — New EMBER2024 Features")

In [ ]:
# General + Strings + Header + Section (best combo)
X_tr_c = np.hstack([X_train_full[:, GENERAL_START:GENERAL_END], X_train_full[:, STRING_START:SECTION_END]])
X_te_c = np.hstack([X_test_full[:,  GENERAL_START:GENERAL_END], X_test_full[:,  STRING_START:SECTION_END]])
run_experiment(X_tr_c, X_te_c, y_train, y_test, get_lgbm(), "LightGBM — General + Strings + Header + Section")

In [ ]:
# EMBER2018-equivalent features
X_tr_old = np.hstack([X_train_full[:, 0:DOS_START], X_train_full[:, DOS_END:DATADIR_START]])
X_te_old = np.hstack([X_test_full[:,  0:DOS_START], X_test_full[:,  DOS_END:DATADIR_START]])
run_experiment(X_tr_old, X_te_old, y_train, y_test, get_lgbm(), "LightGBM — EMBER2018-equivalent Features")